In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Cas n°1 : Économie dans un environnement libre et stable.
# Agents rationnels, information parfaite, pas d'influence extérieure.

In [3]:
def passage(matrice, etat):
    """ Définit aléatoirement le prochain état du marché """
    k = np.random.uniform()
    if k < matrice[etat][0]:
        return 0
    elif k < matrice[etat][0] + matrice[etat][1]:
        return 1
    else:
        return 2

def probapopulation(etat, vecteurpopulation):
    """ Modifie l'opinion de la population selon l'état """
    if etat == 0:
        return vecteurpopulation * 0.8
    if etat == 1:
        return vecteurpopulation
    if etat == 2: 
        return np.array([min(1, e * 1.2) for e in vecteurpopulation])

def prop_investisseurs(etat, vecteurpopulation):
    """ Détermine les investisseurs actifs """
    if etat == 2:
        return np.array([e for e in vecteurpopulation if e > 0.4])
    if etat == 1:
        return np.array([e for e in vecteurpopulation if e > 0.5])
    if etat == 0:
        return np.array([e for e in vecteurpopulation if e < 0.4])
    return np.array([])

def calculer_indice(vecteurindice, vecteur_investisseurs, etat):
    """ Calcule l'indice global du marché """
    for i in range(len(vecteurindice)):
        res = 0
        n_inv = len(vecteur_investisseurs)
        if etat == 0:
            res = -np.sum(np.random.normal(0.001, 0.0001, n_inv))
        elif etat == 2:
            res = np.sum(np.random.normal(0.001, 0.0001, n_inv))
        elif etat == 1:
            res = np.sum(np.random.normal(0, 0.0001, n_inv))
        
        vecteurindice[i] = max(0.01, vecteurindice[i] + res)

    return np.mean(vecteurindice)

def probamatrice(matrice, etat, indice1, indice0, epsilon):
    """ Ajuste les probabilités de transition """
    variation = (indice1 - indice0) / max(indice0, 0.01)
    
    if abs(variation) < 0.0001:
        return matrice
    
    if indice1 <= 0.01:
        matrice[etat][0] = min(0.99, matrice[etat][0] + epsilon)
        matrice[etat][2] = max(0, matrice[etat][2] - epsilon)
        matrice[etat][1] = 1 - matrice[etat][0] - matrice[etat][2]
        return matrice

    transfert = min(0.1, abs(epsilon * variation))
    if variation > 0:
        perte = min(matrice[etat][0], transfert)
        matrice[etat][0] -= perte
        matrice[etat][2] += perte
    else:
        perte = min(matrice[etat][2], transfert)
        matrice[etat][0] += perte
        matrice[etat][2] -= perte

    matrice[etat][1] = 1 - matrice[etat][0] - matrice[etat][2]
    return matrice

In [4]:
def fonction_cas1(matrice, etat, vecteurpopulation, vecteurindice, n, epsilon):
    vecteur_indices = [1.0]
    matrice_pop = []
    vecteur_etats = [etat]
    
    for i in range(n):
        nvetat = passage(matrice, vecteur_etats[i])
        vecteurpopulation = probapopulation(nvetat, vecteurpopulation)
        matrice_pop.append(vecteurpopulation.copy())
        
        vecteur_investisseurs = prop_investisseurs(nvetat, vecteurpopulation)
        nouvel_indice = calculer_indice(vecteurindice, vecteur_investisseurs, nvetat)
        vecteur_indices.append(nouvel_indice)
        
        matrice = probamatrice(matrice, nvetat, vecteur_indices[i], vecteur_indices[i+1], epsilon)
        vecteur_etats.append(nvetat)
        
    return matrice, vecteur_etats, vecteur_indices, matrice_pop

In [5]:
# Initialisation des paramètres de simulation
population = 100
titres2 = 10
matrice_init = [[0.99, 0.01, 0.0], [0.05, 0.9, 0.05], [0.0, 0.01, 0.99]]

vecteurpopulation1 = np.random.normal(0.5, 0.15, population)
vecteurindices1 = np.ones(titres2) # On commence à 1 pour la stabilité

In [6]:
res = fonction_cas1(matrice_init, 0, vecteurpopulation1, vecteurindices1, 20, 0.01)
print("Matrice finale :", res[0])